# Training 10 · Benchmark attackers against published predictors

Five attackers per published predictor (SSL-MOS, UTMOS22, NISQA, DNSMOS), 500 BVCC test utterances.
About 30 min per predictor. Evaluation-only counterpart: `tables/table_bench_published.ipynb`.

In [1]:
import os, sys
from pathlib import Path

# Find the repository root from wherever the notebook runs (notebooks/ or a subfolder).
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "hackpatch").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("start the notebook from inside the repository")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters
# LAUNCH = False only shows what would run (the job script of the first run); True starts the training.
LAUNCH = os.environ.get("NB_LAUNCH", "0") == "1"
# RUN = "slurm": one GPU job per run (sbatch); "local": in this kernel (needs a GPU, runs sequentially).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = [k for k, v in reg.items() if v["kind"] in ['bench']]
print(len(KEYS), "runs:", ", ".join(KEYS))

4 runs: bench_sslmos, bench_utmos, bench_nisqa, bench_dnsmos


In [4]:
# Stored configuration of the first run (checkpoint paths point at the local released copies).
from omegaconf import OmegaConf
print(OmegaConf.to_yaml(repro.load_config(KEYS[0])))

config_name: metric_bench
task: metric_bench
phase: main
seed: 0
run_name: metric_bench_sslmos_s0__2026-09-16_05-51-23
target_name: sslmos
target_ckpt: /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/checkpoints/r0_s0.checkpoints.best.pt
model:
  pretrained: facebook/wav2vec2-base
  freeze_feature_encoder: true
  freeze_encoder: false
data:
  corpus: bvcc
  max_seconds: 6.0
  num_workers: 8
attack:
  codec:
    name: encodec_24khz
    quantize: false
    encodec_bandwidth: 24.0
  policy:
    hidden: 256
    layers: 4
    kernel: 5
    init_log_std: -1.0
  action_hop: 0
  budget_mode: sphere
  eps: 0.5
  budget_penalty: 0.0
  group_size: 8
  batch_size: 8
  crop_seconds: 3.0
  steps: 2000
  lr: 0.001
  ppo_epochs: 2
  clip: 0.2
  adv_eps: 0.0001
  grad_clip: 1.0
  log_every: 10
  apply:
    n_samples: 3
    batch_size: 16
    chunk_utts: 512
    max_seconds: 20.0
detect:
  tau: 0.3
  measures:
  - estoi
  - sisdr
  - wer
  - sbs
  margins:
    estoi: 0.02
    sis

In [5]:
if LAUNCH:
    repro.run_keys(KEYS, "retrain", RUN)  # outputs: {scratch}/projects/audio_assessment_hackability/retrain/<key>/
else:
    print("LAUNCH=False: nothing started. Job script for", KEYS[0], "->")
    repro.submit(KEYS[0], "retrain", dry_run=True)

LAUNCH=False: nothing started. Job script for bench_sslmos ->
#!/bin/bash
#SBATCH --account=PAS2966
#SBATCH --partition=nextgen
#SBATCH --job-name=retrain_bench_sslmos
#SBATCH --output=/fs/scratch/PAS2301/alialavi/projects/audio_assessment_hackability/retrain/_logs/%x_%j.out
#SBATCH --time=02:00:00
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=12
#SBATCH --mem=64G
#SBATCH --gpus-per-node=1

set -euo pipefail
cd /users/PAS2301/alialavi/projects/audio_assessment_hackability
source scripts/env.sh
python scripts/repro_run.py --mode retrain --key bench_sslmos



## Retrained vs original

In [6]:
import json
released = repro.fetch_results()
rows = {}
for k in KEYS:
    for src, d in (("original", released / Path(reg[k]["run_dir"]).name), ("retrained", repro.RUNS / "retrain" / k)):
        f = d / "results/bench_summary.json"
        if f.exists():
            rows[(k, src)] = pd.json_normalize(json.loads(f.read_text())).iloc[0]
df = pd.DataFrame(rows).T
df[["target", "asr_ref_recon_mean", "asr_ref_recon_max", "gain_ref_recon_mean", "recon_gain_mean"]] if len(df) else "no runs found"

,,target,asr_ref_recon_mean,asr_ref_recon_max,gain_ref_recon_mean,recon_gain_mean
bench_sslmos,original,sslmos,0.2048,0.252,0.191362,-0.099547
bench_utmos,original,utmos,0.0572,0.076,0.143828,-0.068687
bench_nisqa,original,nisqa,0.896,0.916,0.954347,0.139453
bench_dnsmos,original,dnsmos,0.1356,0.2,0.124104,0.015846
